## Batch Gradiant Descend 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from sklearn.metrics import root_mean_squared_error, r2_score

df=pd.read_csv("/Users/mac/Desktop/Machine Learning/ai-roadmap/content/ml/supervised/03 gradient descent/data/advertising.csv")

X=df.drop(columns=["Sales"]).values
y=df["Sales"].values

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

## Batch Gradiant Descend ##

In [ ]:
class BatchGradiantDescend:
    def __init__(self, lr, max_iter, tol):
        self.lr=lr
        self.max_iter=max_iter
        self.tol=tol
        
        self.coef_=None
        self.intercept_=None
        
        self.lossi=[]
        
    def fit(self, X, y):
        X=np.array(X)
        y=np.array(y)
        n=X.shape[0]
        
        X=np.c_[np.ones(n), X]
        
        n_features=X.shape[1]
        
        # 1. Initialize coefficients randomly
        coef=np.random.randn(n_features)
        
        converged=False
        
        for i in range(self.max_iter):
            y_pred=X @ coef
            e=y-y_pred
            
            loss=root_mean_squared_error(y, y_pred)
            self.lossi.append(loss)
            
            #2.calculate gradients
            coef_grad=(-2/n)* X.T @ e
            
            if np.linalg.norm(self.lr * coef_grad)< self.tol:
                converged=True
                break
            
            # 3. Adjust coefficients accordingly
            coef=coef-self.lr*coef_grad
            
        if not converged:
            print(f"Warning: did not converged after {self.max_iter} steps")
            
        # Repeat this procces self.max_iter times
        self.coef_=coef[1:]
        self.intercept_=coef[0]
        self.step=i  #type: ignore
        return self
    
    def predict(self, X):
        X=np.array(X)
        return X @ self.coef_ +self.intercept_    # type: ignore
    
    def score(self, X, y):
        y_pred=self.predict(X)
        return r2_score(y, y_pred)
            
        

In [ ]:
gd=BatchGradiantDescend(lr=0.01, max_iter=1000, tol=0.001)
gd.fit(X_scaled, y)

#Prediction
gd.predict(X_scaled)

print("R2 error(score) with batch gradient descend:", gd.score(X_scaled, y))

## Multiple linear regresion

In [ ]:
lin_reg=LinearRegression()
lin_reg.fit(X_scaled, y)

y_pred=lin_reg.predict(X_scaled)

print("Prediction result by multiple linear regression:", r2_score(y, y_pred))



## Stochastic Gradient Descend 

In [ ]:
class StochasticGradiantDescend:
    
    def __init__(self, lr, max_iter, tol):
        self.lr=lr
        self.max_iter=max_iter
        self.tol=tol
        
        self.coef_=None
        self.intercept_=None
        
        self.lossi=[]
        
    def fit(self, X, y):
        X=np.array(X)
        y=np.array(y)
        n=X.shape[0]
        
        X = np.c_[np.ones(n), X]
        
        n_features = X.shape[1]
        
        # 1. Initialize coefficients randomly
        coef = np.random.randn(n_features)
        
        converged=False
        
        for epoch in range(self.max_iter):
            for i in range(n):
                y_pred=X[i] @ coef
                
                e=y[i]-y_pred
                
                loss=root_mean_squared_error([y[i]], [y_pred])
                self.lossi.append(loss)
                
                coef_grad=(-2)*X[i].T* e
                
                if np.linalg.norm(self.lr * coef_grad)<self.tol:
                    converged=True
                    break
                
                coef=coef-self.lr*coef_grad
                
        if not converged:
            print(f"Warning: Did not converge after {self.max_iter} steps.")
            
        
        self.coef_ = coef[1:]
        self.intercept_ = coef[0]
        self.step = epoch
        return self
    
    def predict(self, X):
        X=np.array(X)
        return X @ self.coef_+self.intercept_ #type: ignore

    def score(self, X, y):
        y_pred=self.predict(X)
        return r2_score(y, y_pred)
        
        

In [ ]:
sgd=StochasticGradiantDescend(lr=0.01, max_iter=10, tol=0.001)
sgd.fit(X_scaled, y)

print("R2 error(score) with stochastic gradiant descend:", sgd.score(X_scaled, y))

## Mini Batch Gradiant Descend

In [ ]:
class MiniBAtchGradientDescend:
    def __init__(self,lr=0.01, max_iter=100, batch_size=32):
        self.lr = lr
        self.max_iter = max_iter
        self.batch_size = batch_size
        
        self.coef_ = None
        self.intercept_ = None
        self.lossi = []
        
    def fit(self, X, y):
        X=np.array(X)
        y=np.array(y)
        n=X.shape[0]
        
        X = np.c_[np.ones(n), X]
        n_features = X.shape[1]
        
        #Initialize Weight
        coef=np.random.randn(n_features)
        
        for epoch in range(self.max_iter):
            #Shuffle data
            inx=np.random.permutation(n)
            X=X[inx]
            y=y[inx]
            
            for i in range(0, n, self.batch_size):
                X_batch=X[i:i+self.batch_size]
                y_batch=y[i:i+self.batch_size]
                
                y_pred=X_batch @ coef
                
                e=y_batch-y_pred
                
                grad=(-2/len(X_batch)) * X_batch.T @ e
                
                coef=coef-self.lr*grad
                
            y_full_pred=X@coef
            loss=root_mean_squared_error(y, y_full_pred)
            self.lossi.append(loss)
            
        self.intercept_=coef[0]
        self.coef_=coef[1:]
        
        return self
    
    def predict(self, X):
        X=np.array(X)
        return X@ self.coef_ + self.intercept_  #type:ignore
    
    def score(self, X, y):
        y_pred=self.predict(X)
        return r2_score(y, y_pred)

In [ ]:
mbgd=MiniBAtchGradientDescend(lr=0.01, max_iter=100, batch_size=32)
mbgd.fit(X_scaled, y)

print("R2 score (Mini-batch):", mbgd.score(X_scaled, y))

$$ Plot

In [ ]:

plt.plot(mbgd.lossi)
plt.xlabel("Epoch")
plt.ylabel("RMSE Loss")
plt.title("Mini-Batch Gradient Descent Convergence")
plt.grid()
plt.show()

In [ ]:
plt.figure()

plt.plot(gd.lossi, label="Batch GD")
plt.plot(sgd.lossi, label="SGD")
plt.plot(mbgd.lossi, label="Mini-Batch GD")

plt.xlabel("Iterations / Epochs")
plt.ylabel("RMSE Loss")
plt.title("Convergence Comparison")

plt.legend()
plt.grid()
plt.show()